In [ ]:
# Imports & global config
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:

def record_results(
    model_name,
    y_true,
    y_pred,
    mean_ci,
    ci_low,
    ci_high,
    delta_acc=None,
    delta_ci_low=None,
    delta_ci_high=None,
    mcnemar_b=None,
    mcnemar_c=None,
    mcnemar_p=None,
):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),

        # non-paired descriptive CI
        "Acc_CI_mean": mean_ci,
        "Acc_CI_low": ci_low,
        "Acc_CI_high": ci_high,

        # paired vs human (filled later)
        "Δ_Accuracy_vs_Human": delta_acc,
        "Δ_CI_low": delta_ci_low,
        "Δ_CI_high": delta_ci_high,
        "McNemar_b": mcnemar_b,
        "McNemar_c": mcnemar_c,
        "McNemar_p": mcnemar_p,
        "Significant_0.05": (
            None if mcnemar_p is None else mcnemar_p < 0.05
        ),
    }


In [ ]:
consult_df = pd.read_csv("/content/Human_vs_ground_truth.csv")

# Normalize text labels
label_map = {"Plaintiff": 1, "Defendant": 0}

consult_df["Observed"] = consult_df["Observed"].map(label_map)
consult_df["Consultant2"] = consult_df["Consultant2"].map(label_map)
consult_df["Consultant3"] = consult_df["Consultant3"].map(label_map)
consult_df["Consultant4"] = consult_df["Consultant4"].map(label_map)

# Inspect
consult_df.head()


In [ ]:
# Majority vote across Consultant2, Consultant3, Consultant4
consult_df["MajorityVote"] = (
    consult_df[["Consultant2", "Consultant3", "Consultant4"]].sum(axis=1) >= 2
).astype(int)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_true = consult_df["Observed"]
y_pred = consult_df["MajorityVote"]

acc_consult = accuracy_score(y_true, y_pred)
prec_consult = precision_score(y_true, y_pred)
rec_consult = recall_score(y_true, y_pred)
f1_consult = f1_score(y_true, y_pred)

print("Human expert Accuracy:", acc_consult)
print("Precision:", prec_consult)
print("Recall:", rec_consult)
print("F1:", f1_consult)


In [ ]:

# Bootstrap CI for majority vote predictions
def bootstrap_consultant_ci(y_true, y_pred, B=5000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    accs = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        accs.append(accuracy_score(y_true.iloc[idx], y_pred.iloc[idx]))
    return np.mean(accs), np.percentile(accs, 2.5), np.percentile(accs, 97.5)

mean_ci_c, ci_low_c, ci_high_c = bootstrap_consultant_ci(y_true, y_pred)

print(f"[Consultants] 95% CI: mean={mean_ci_c:.3f}, ({ci_low_c:.3f}, {ci_high_c:.3f})")


In [ ]:
# Results container
results = []  # each element: dict(Model, Accuracy, CI_mean, CI_low, CI_high)


# Append experts results to results DataFrame
human_results = record_results(
    "Human Expert",
    y_true, y_pred,
    mean_ci_c, ci_low_c, ci_high_c
)


In [ ]:
def paired_bootstrap_accuracy_diff(
    y_true,
    y_pred_model,
    y_pred_human,
    B=5000,
    seed=42
):
    """
    Paired bootstrap CI for accuracy difference:
    Acc(model) - Acc(human)
    """
    rng = np.random.default_rng(seed)
    n = len(y_true)
    diffs = []

    for _ in range(B):
        idx = rng.integers(0, n, n)

        acc_model = accuracy_score(
            y_true.iloc[idx],
            y_pred_model.iloc[idx]
        )
        acc_human = accuracy_score(
            y_true.iloc[idx],
            y_pred_human.iloc[idx]
        )

        diffs.append(acc_model - acc_human)

    diffs = np.array(diffs)

    return {
        "diff_mean": diffs.mean(),
        "diff_low": np.percentile(diffs, 2.5),
        "diff_high": np.percentile(diffs, 97.5),
    }


In [ ]:
def mcnemar_test(y_true, y_pred_model, y_pred_human):
    """
    McNemar test comparing model vs human.
    """
    model_correct = y_pred_model == y_true
    human_correct = y_pred_human == y_true

    table = pd.crosstab(
        model_correct,
        human_correct
    )

    # Ensure full 2x2 table
    table = table.reindex(
        index=[False, True],
        columns=[False, True],
        fill_value=0
    )

    if table.loc[True, False] + table.loc[False, True] < 25:
      result = mcnemar(table, exact=True)
    else:
      result = mcnemar(table, exact=False, correction=True)

    return {
        "b": table.loc[True, False],   # model correct, human wrong
        "c": table.loc[False, True],   # model wrong, human correct
        "p_value": result.pvalue
    }


In [ ]:
# Load data

# Adjust path as needed
df = pd.read_csv("/content/data_employment_v2_filtered_encoded_12102025.csv")
df = df.dropna(how="all").reset_index(drop=True)

# Clean column names
df.columns = (
    df.columns.str.replace(r"[\n\r\t]", " ", regex=True)
              .str.replace(r"[()\[\]{}]", "", regex=True)
              .str.replace(r"\s+", "_", regex=True)
              .str.strip("_")
)

# Sex → ordinal
sex_clean = df["What_is_your_sex?"].astype(str).str.strip().str.lower()
df["What_is_your_sex?_ord"] = (
    sex_clean.map({"female": 0, "male": 1})
    .fillna(2)  # 2 for "Other / prefer not to say / anything else"
)

# Race → one-hot
df = pd.get_dummies(
    df,
    columns=["Which_of_the_following_best_matches_your_race?"],
    prefix="Race"
)

selected_columns_df = df.iloc[:, 28:58]

# Target is the 23rd column in this slice (0-based index 22)
target_col = selected_columns_df.columns[22]
print("Slice shape:", selected_columns_df.shape)
print("Target column:", target_col)

# Features and target
X = selected_columns_df.drop(columns=[target_col])
y = selected_columns_df[target_col]

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts(normalize=True))


In [ ]:
for idx, row in X.head(5).iterrows():
    print(f"Row {idx}:")
    for col, val in row.items():
        print(f"  {col}: {val}")
    print("-" * 40)

In [ ]:

# Train/test split using fixed manager indices

manager_indices = [
    11, 13, 15, 17, 22, 33, 34, 43, 55, 68, 72, 73, 76, 82, 85,
    87, 91, 92, 93, 104, 108, 109, 119, 121, 128, 129, 135, 141,
    144, 145, 147, 153, 155, 157, 166, 167, 170, 171, 172, 174,
    176, 178, 184, 189, 203, 204, 207, 208, 209, 212
]

test_mask = X.index.isin(manager_indices)
X_train, y_train = X[~test_mask], y[~test_mask]
X_test, y_test = X[test_mask], y[test_mask]

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))


In [ ]:

# Utility: Bootstrap 95% CI for accuracy
def bootstrap_ci_accuracy(model, X, y, B=5000, seed=RANDOM_STATE):
    """
    Nonparametric bootstrap CI for accuracy on a fixed test set.
    """
    rng = np.random.default_rng(seed)
    n = len(y)
    accs = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        Xb = X.iloc[idx]
        yb = y.iloc[idx]
        y_pred_b = model.predict(Xb)
        accs.append(accuracy_score(yb, y_pred_b))
    accs = np.array(accs)
    mean_acc = accs.mean()
    low = np.percentile(accs, 2.5)
    high = np.percentile(accs, 97.5)
    return mean_acc, low, high


def bootstrap_ci_stacking(xgb_model, meta_model, X, y, B=5000, seed=RANDOM_STATE):
    """
    Bootstrap CI for stacking model:
    - XGB produces meta-probabilities
    - Logistic regression consumes them
    """
    rng = np.random.default_rng(seed)
    n = len(y)
    accs = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        Xb = X.iloc[idx]
        yb = y.iloc[idx]
        meta_probs = xgb_model.predict_proba(Xb)[:, 1].reshape(-1, 1)
        y_pred_b = meta_model.predict(meta_probs)
        accs.append(accuracy_score(yb, y_pred_b))
    accs = np.array(accs)
    mean_acc = accs.mean()
    low = np.percentile(accs, 2.5)
    high = np.percentile(accs, 97.5)
    return mean_acc, low, high


In [ ]:

# Baseline XGB

xgb_base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=RANDOM_STATE
)

param_grid_base = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

cv_base = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

gs_base = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_base,
    cv=cv_base,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)

gs_base.fit(X_train, y_train)
best_xgb_base = gs_base.best_estimator_

y_pred = best_xgb_base.predict(X_test)


# CI for accuracy
mean_ci, ci_low, ci_high = bootstrap_ci_accuracy(best_xgb_base, X_test, y_test)

# Append metrics
results.append(record_results(
    "XGB Baseline",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
))

print(record_results(
    "XGB Baseline",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
))


In [ ]:
# Extended / Tuned XGB

from xgboost import XGBClassifier
xgb_ext = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=RANDOM_STATE
)

param_dist_ext = {
    "n_estimators": [200, 300, 400, 600],
    "max_depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "gamma": [0, 0.1, 0.2],
}

cv_ext = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rs_ext = RandomizedSearchCV(
    estimator=xgb_ext,
    param_distributions=param_dist_ext,
    n_iter=60,
    scoring="accuracy",
    cv=cv_ext,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)

rs_ext.fit(X_train, y_train)
best_xgb_ext = rs_ext.best_estimator_

y_pred = best_xgb_ext.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("\n[Extended XGB] Test accuracy:", acc)
print("[Extended XGB] Best params:", rs_ext.best_params_)
print("[Extended XGB] Classification report:")
print(classification_report(y_test, y_pred, digits=3))

mean_ci, ci_low, ci_high = bootstrap_ci_accuracy(best_xgb_ext, X_test, y_test)
print(f"[Extended XGB] 95% CI: mean={mean_ci:.3f}, ({ci_low:.3f}, {ci_high:.3f})")


# Append metrics
results.append(record_results(
    "XGB Extended",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
))

print(record_results(
    "XGB Extended",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
))

In [ ]:
# Monotonic XGB (all +1 constraints, exploratory)

mono_constraints = "(" + ",".join(["1"] * X_train.shape[1]) + ")"

xgb_mono = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    monotone_constraints=mono_constraints,
)

xgb_mono.fit(X_train, y_train)

y_pred = xgb_mono.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("\n[Monotonic XGB] Test accuracy:", acc)
print("[Monotonic XGB] Classification report:")
print(classification_report(y_test, y_pred, digits=3))

mean_ci, ci_low, ci_high = bootstrap_ci_accuracy(xgb_mono, X_test, y_test)
print(f"[Monotonic XGB] 95% CI: mean={mean_ci:.3f}, ({ci_low:.3f}, {ci_high:.3f})")


# Append metrics
results.append(record_results(
    "XGB Monotonic",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
))

print(record_results(
    "XGB Monotonic",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
))

In [ ]:
# One-Hot Encoded XGB (over all feature columns)

X_full_ohe = pd.get_dummies(X, columns=X.columns, drop_first=False)
X_train_ohe = X_full_ohe[~test_mask]
X_test_ohe = X_full_ohe[test_mask]

print("Original X shape:", X.shape)
print("One-hot X shape :", X_full_ohe.shape)

xgb_ohe = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=RANDOM_STATE
)

param_dist_ohe = {
    "n_estimators": [200, 300, 400],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
}

cv_ohe = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rs_ohe = RandomizedSearchCV(
    estimator=xgb_ohe,
    param_distributions=param_dist_ohe,
    n_iter=40,
    scoring="accuracy",
    cv=cv_ohe,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)

rs_ohe.fit(X_train_ohe, y_train)
best_xgb_ohe = rs_ohe.best_estimator_

y_pred = best_xgb_ohe.predict(X_test_ohe)
acc = accuracy_score(y_test, y_pred)
print("\n[OHE XGB] Test accuracy:", acc)
print("[OHE XGB] Best params:", rs_ohe.best_params_)
print("[OHE XGB] Classification report:")
print(classification_report(y_test, y_pred, digits=3))

# Bootstrap CI for OHE uses X_test_ohe
mean_ci, ci_low, ci_high = bootstrap_ci_accuracy(best_xgb_ohe, X_test_ohe, y_test)
print(f"[OHE XGB] 95% CI: mean={mean_ci:.3f}, ({ci_low:.3f}, {ci_high:.3f})")



# Append metrics
results.append(record_results(
    "XGB OHE",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
))

print(record_results(
    "XGB OHE",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
))

In [ ]:

# XGB Stacking: XGB base + Logistic Regression meta

# Use the best baseline XGB as level-1 model (probabilities)
xgb_level1 = best_xgb_base

# Train meta-features on train
train_meta = xgb_level1.predict_proba(X_train)[:, 1].reshape(-1, 1)
test_meta = xgb_level1.predict_proba(X_test)[:, 1].reshape(-1, 1)

meta_model = LogisticRegression(random_state=RANDOM_STATE)
meta_model.fit(train_meta, y_train)

y_pred = meta_model.predict(test_meta)
acc = accuracy_score(y_test, y_pred)
print("\n[XGB Stacked] Test accuracy:", acc)
print("[XGB Stacked] Classification report:")
print(classification_report(y_test, y_pred, digits=3))

mean_ci, ci_low, ci_high = bootstrap_ci_stacking(xgb_level1, meta_model, X_test, y_test)
print(f"[XGB Stacked] 95% CI: mean={mean_ci:.3f}, ({ci_low:.3f}, {ci_high:.3f})")



# Append metrics
results.append(record_results(
    "XGB Stacked",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
))

print(record_results(
    "XGB Stacked",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
))

In [ ]:

# Final results table

df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Accuracy", ascending=False).reset_index(drop=True)
df_results


In [ ]:
# Ground truth from model test set
y_test_aligned = y_test.reset_index(drop=True)

# Human ground truth + predictions (same ordering by construction)
human_true = consult_df["Observed"].reset_index(drop=True)
human_pred = consult_df["MajorityVote"].reset_index(drop=True)

# HARD safety checks
assert len(y_test_aligned) == len(human_pred), "Length mismatch"
assert (y_test_aligned.values == human_true.values).all(), \
       "Ground-truth mismatch between model test set and consultant CSV!"


In [ ]:
model_predictions = {
    "XGB Baseline": best_xgb_base.predict(X_test),
    "XGB Extended": best_xgb_ext.predict(X_test),
    "XGB Monotonic": xgb_mono.predict(X_test),
    "XGB OHE": best_xgb_ohe.predict(X_test_ohe),
    "XGB Stacked": meta_model.predict(test_meta),
}

In [ ]:
for i, row in df_results.iterrows():
    model = row["Model"]

    if model == "Human Expert":
        continue

    y_model = pd.Series(model_predictions[model]).reset_index(drop=True)

    boot = paired_bootstrap_accuracy_diff(
        y_true=y_test_aligned,
        y_pred_model=y_model,
        y_pred_human=human_pred,
    )

    mc = mcnemar_test(
        y_true=y_test_aligned,
        y_pred_model=y_model,
        y_pred_human=human_pred,
    )

    df_results.loc[i, "Δ_Accuracy_vs_Human"] = boot["diff_mean"]
    df_results.loc[i, "Δ_CI_low"] = boot["diff_low"]
    df_results.loc[i, "Δ_CI_high"] = boot["diff_high"]
    df_results.loc[i, "McNemar_b"] = mc["b"]
    df_results.loc[i, "McNemar_c"] = mc["c"]
    df_results.loc[i, "McNemar_p"] = mc["p_value"]
    df_results.loc[i, "Significant_0.05"] = mc["p_value"] < 0.05


In [ ]:
df_results

In [ ]:

# Barplot with 95% CI and Save to File
plt.figure(figsize=(12,6))

sns.barplot(
    data=df_results,
    x="Model",
    y="Accuracy",
    color="skyblue",
    edgecolor="black"
)

# Add CI error bars
for i, row in df_results.reset_index(drop=True).iterrows():
    plt.errorbar(
        i,
        row["Accuracy"],
        yerr=[[row["Accuracy"] - row["Acc_CI_low"]],
              [row["Acc_CI_high"] - row["Accuracy"]]],
        fmt='none',
        ecolor='black',
        capsize=5,
        linewidth=1.5
    )

plt.title("XGBoost Models vs Human Expert\nAccuracy with 95% CI", fontsize=14)
plt.ylabel("Accuracy", fontsize=12)
plt.xticks(rotation=40, ha="right")
plt.ylim(0.4, 1.0)
plt.tight_layout()

# ---- SAVE THE PLOT ----
plt.savefig("xgb_vs_consultants_accuracy_CI.png", dpi=300, bbox_inches='tight')
plt.savefig("xgb_vs_consultants_accuracy_CI.pdf", bbox_inches='tight')
plt.savefig("xgb_vs_consultants_accuracy_CI.svg", bbox_inches='tight')

plt.show()

print("Plot saved as:")
print(" - xgb_vs_consultants_accuracy_CI.png")
print(" - xgb_vs_consultants_accuracy_CI.pdf")
print(" - xgb_vs_consultants_accuracy_CI.svg")


In [ ]:
plt.figure(figsize=(10,5))
sns.heatmap(
    df_results.set_index("Model")[["Accuracy","Precision","Recall","F1"]],
    annot=True, cmap="Blues", fmt=".2f"
)
plt.title("Model Performance Metrics (Accuracy, Precision, Recall, F1)")
plt.tight_layout()
plt.savefig("xgb_metric_heatmap.png", dpi=300)
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold

results_rf = []   # separate from results for XGB
RANDOM_STATE = 42


In [ ]:

# RF Baseline

rf_base = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

param_grid_rf_base = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
}

cv_rf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

gs_rf_base = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid_rf_base,
    cv=cv_rf,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)

gs_rf_base.fit(X_train, y_train)
best_rf_base = gs_rf_base.best_estimator_

y_pred = best_rf_base.predict(X_test)
mean_ci, ci_low, ci_high = bootstrap_ci_accuracy(best_rf_base, X_test, y_test)

rf_baseline_results = record_results(
    "RF Baseline",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
)

results_rf.append(rf_baseline_results)

print("[RF Baseline] Best params:", gs_rf_base.best_params_)
print("[RF Baseline] Metrics:", rf_baseline_results)


In [ ]:

# RF Extended / Tuned

rf_ext = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

param_dist_rf_ext = {
    "n_estimators": [200, 400, 600],
    "max_depth": [5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
    "class_weight": [None, "balanced"],
}

rs_rf_ext = RandomizedSearchCV(
    estimator=rf_ext,
    param_distributions=param_dist_rf_ext,
    n_iter=60,
    cv=cv_rf,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE,
)

rs_rf_ext.fit(X_train, y_train)
best_rf_ext = rs_rf_ext.best_estimator_

y_pred = best_rf_ext.predict(X_test)
mean_ci, ci_low, ci_high = bootstrap_ci_accuracy(best_rf_ext, X_test, y_test)

rf_ext_results = record_results(
    "RF Extended",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
)

results_rf.append(rf_ext_results)

print("[RF Extended] Best params:", rs_rf_ext.best_params_)
print("[RF Extended] Metrics:", rf_ext_results)


In [ ]:

# RF F1-Optimized (balanced emphasis)

rf_f1 = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

param_dist_rf_f1 = {
    "n_estimators": [200, 400],
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"],
}

rs_rf_f1 = RandomizedSearchCV(
    estimator=rf_f1,
    param_distributions=param_dist_rf_f1,
    n_iter=40,
    cv=cv_rf,
    scoring="f1",     # <--- key difference
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE,
)

rs_rf_f1.fit(X_train, y_train)
best_rf_f1 = rs_rf_f1.best_estimator_

y_pred = best_rf_f1.predict(X_test)
mean_ci, ci_low, ci_high = bootstrap_ci_accuracy(best_rf_f1, X_test, y_test)

rf_f1_results = record_results(
    "RF F1-Optimized",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
)

results_rf.append(rf_f1_results)

print("[RF F1-Optimized] Best params:", rs_rf_f1.best_params_)
print("[RF F1-Optimized] Metrics:", rf_f1_results)


In [ ]:
# RF Stacked: RF base + Logistic Regression meta-model


rf_level1 = best_rf_base  # use baseline RF as base predictor

train_meta = rf_level1.predict_proba(X_train)[:, 1].reshape(-1, 1)
test_meta  = rf_level1.predict_proba(X_test)[:, 1].reshape(-1, 1)

meta_rf = LogisticRegression(random_state=RANDOM_STATE)
meta_rf.fit(train_meta, y_train)

y_pred = meta_rf.predict(test_meta)

# Bootstrap for stacking: we must rewrap to recompute meta probs inside each resample.
def bootstrap_ci_stacking_rf(base_model, meta_model, X, y, B=5000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y)
    accs = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        Xb = X.iloc[idx]
        yb = y.iloc[idx]
        meta_probs = base_model.predict_proba(Xb)[:, 1].reshape(-1, 1)
        y_pred_b = meta_model.predict(meta_probs)
        accs.append(accuracy_score(yb, y_pred_b))
    accs = np.array(accs)
    return accs.mean(), np.percentile(accs, 2.5), np.percentile(accs, 97.5)

mean_ci, ci_low, ci_high = bootstrap_ci_stacking_rf(rf_level1, meta_rf, X_test, y_test)

rf_stack_results = record_results(
    "RF Stacked",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
)

results_rf.append(rf_stack_results)

print("[RF Stacked] Metrics:", rf_stack_results)


In [ ]:
# y_true, y_pred for Human Expert majority vote
# mean_ci_c, ci_low_c, ci_high_c from bootstrap_consultant_ci


y_true = consult_df["Observed"]
y_pred = consult_df["MajorityVote"]

mean_ci_c, ci_low_c, ci_high_c = bootstrap_consultant_ci(y_true, y_pred)

print(f"[Consultants] 95% CI: mean={mean_ci_c:.3f}, ({ci_low_c:.3f}, {ci_high_c:.3f})")

human_rf = record_results(
    "Human Expert",
    y_true, y_pred,     # from consultant_df
    mean_ci_c, ci_low_c, ci_high_c
)

results_rf.append(human_rf)
print("[Human Expert] Metrics:", human_rf)


In [ ]:

# TRUE STACKING: RF + XGB(OHE) → Logistic Regression (with threshold tuning)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np


# Build meta-features (probabilities from base models)

# Base model probabilities on TRAIN
rf_train_proba  = best_rf_base.predict_proba(X_train)[:, 1]
xgb_train_proba = best_xgb_ohe.predict_proba(X_train_ohe)[:, 1]

train_meta = np.column_stack([
    rf_train_proba,
    xgb_train_proba,
])

# Base model probabilities on TEST
rf_test_proba  = best_rf_base.predict_proba(X_test)[:, 1]
xgb_test_proba = best_xgb_ohe.predict_proba(X_test_ohe)[:, 1]

test_meta = np.column_stack([
    rf_test_proba,
    xgb_test_proba,
])


# Train meta-learner
meta_model = LogisticRegression(
    C=1.0,
    penalty="l2",
    solver="lbfgs",
    random_state=RANDOM_STATE
)

meta_model.fit(train_meta, y_train)


#  Threshold optimization on TRAIN (accuracy)
def find_best_threshold(y_true, y_proba):
    thresholds = np.linspace(0.1, 0.9, 81)
    accs = [
        accuracy_score(y_true, (y_proba >= t).astype(int))
        for t in thresholds
    ]
    best_idx = np.argmax(accs)
    return thresholds[best_idx], accs[best_idx]

train_meta_proba = meta_model.predict_proba(train_meta)[:, 1]
best_threshold, best_train_acc = find_best_threshold(y_train, train_meta_proba)

print(f"[Stacked] Best threshold (train): {best_threshold:.2f}")
print(f"[Stacked] Train accuracy @ threshold: {best_train_acc:.3f}")


# Evaluate on TEST

test_meta_proba = meta_model.predict_proba(test_meta)[:, 1]
y_pred = (test_meta_proba >= best_threshold).astype(int)

acc = accuracy_score(y_test, y_pred)

print("\n[RF + XGB Stacked] Test accuracy:", acc)
print("[RF + XGB Stacked] Classification report:")
print(classification_report(y_test, y_pred, digits=3))


#  Bootstrap 95% CI for stacked model (probability-based)
def bootstrap_ci_stacked_probs(y_true, proba, threshold, B=5000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    accs = []

    for _ in range(B):
        idx = rng.integers(0, n, n)
        yb = y_true.iloc[idx]
        pb = proba[idx]
        accs.append(
            accuracy_score(yb, (pb >= threshold).astype(int))
        )

    accs = np.array(accs)
    return accs.mean(), np.percentile(accs, 2.5), np.percentile(accs, 97.5)

mean_ci, ci_low, ci_high = bootstrap_ci_stacked_probs(
    y_test.reset_index(drop=True),
    test_meta_proba,
    best_threshold,
)

print(f"[RF + XGB Stacked] 95% CI: mean={mean_ci:.3f}, ({ci_low:.3f}, {ci_high:.3f})")


# Record results
results_rf.append(record_results(
    "RF + XGB Stacked",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
))

print(record_results(
    "RF + XGB Stacked",
    y_test, y_pred,
    mean_ci, ci_low, ci_high
))


In [ ]:
df_results_rf = pd.DataFrame(results_rf).sort_values("Accuracy", ascending=False)
df_results_rf


In [ ]:
# Ground truth from model test set
y_test_aligned = y_test.reset_index(drop=True)

# Human ground truth + predictions (same ordering by construction)
human_true = consult_df["Observed"].reset_index(drop=True)
human_pred = consult_df["MajorityVote"].reset_index(drop=True)

# HARD safety checks
assert len(y_test_aligned) == len(human_pred), "Length mismatch"
assert (y_test_aligned.values == human_true.values).all(), \
       "Ground-truth mismatch between model test set and consultant CSV!"

# --- RF-only stacked predictions (meta_rf expects 1 feature: RF proba) ---
rf_stacked_test_meta = best_rf_base.predict_proba(X_test)[:, 1].reshape(-1, 1)
rf_stacked_pred = meta_rf.predict(rf_stacked_test_meta)

# --- RF + XGB stacked predictions (recompute here) ---
rf_xgb_stacked_pred = (meta_model.predict_proba(test_meta)[:, 1] >= best_threshold).astype(int)

rf_model_predictions = {
    "RF Baseline": best_rf_base.predict(X_test),
    "RF Extended": best_rf_ext.predict(X_test),
    "RF F1-Optimized": best_rf_f1.predict(X_test),
    "RF Stacked": rf_stacked_pred,
    "RF + XGB Stacked": rf_xgb_stacked_pred,
}

In [ ]:
for i, row in df_results_rf.iterrows():
    model = row["Model"]

    if model == "Human Expert":
        continue

    y_model = pd.Series(rf_model_predictions[model]).reset_index(drop=True)

    boot = paired_bootstrap_accuracy_diff(
        y_true=y_test_aligned,
        y_pred_model=y_model,
        y_pred_human=human_pred,
    )

    mc = mcnemar_test(
        y_true=y_test_aligned,
        y_pred_model=y_model,
        y_pred_human=human_pred,
    )

    df_results_rf.loc[i, "Δ_Accuracy_vs_Human"] = boot["diff_mean"]
    df_results_rf.loc[i, "Δ_CI_low"] = boot["diff_low"]
    df_results_rf.loc[i, "Δ_CI_high"] = boot["diff_high"]
    df_results_rf.loc[i, "McNemar_b"] = mc["b"]
    df_results_rf.loc[i, "McNemar_c"] = mc["c"]
    df_results_rf.loc[i, "McNemar_p"] = mc["p_value"]
    df_results_rf.loc[i, "Significant_0.05"] = mc["p_value"] < 0.05

In [ ]:
df_results_rf

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=df_results_rf,
    x="Model",
    y="Accuracy",
    color="lightgreen",
    edgecolor="black"
)

for i, row in df_results_rf.reset_index(drop=True).iterrows():
    plt.errorbar(
        i,
        row["Accuracy"],
        yerr=[[row["Accuracy"] - row["Acc_CI_low"]],
              [row["Acc_CI_high"] - row["Accuracy"]]],
        fmt='none',
        ecolor='black',
        capsize=5,
        linewidth=1.5
    )

plt.title("Random Forest Variants vs Human Expert\nAccuracy with 95% Bootstrap CI")
plt.ylabel("Test Accuracy")
plt.xticks(rotation=40, ha="right")
plt.ylim(0.4, 1.0)
plt.tight_layout()

plt.savefig("rf_vs_consultants_accuracy_CI.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
plt.figure(figsize=(8,4))
sns.heatmap(
    df_results_rf.set_index("Model")[["Accuracy","Precision","Recall","F1"]],
    annot=True, cmap="Greens", fmt=".2f"
)
plt.title("Random Forest Models vs Human Expert\nPerformance Metrics")
plt.tight_layout()
plt.savefig("rf_metric_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:

# Collect predictions from best models + human

# 1) XGBoost (OHE) predictions on the test set
xgb_ohe_pred = best_xgb_ohe.predict(X_test_ohe)
xgb_ohe_proba = best_xgb_ohe.predict_proba(X_test_ohe)[:, 1]

# 2) Random Forest predictions on the test set
rf_pred = best_rf_base.predict(X_test)
rf_proba = best_rf_base.predict_proba(X_test)[:, 1]

# 3) Human majority vote predictions on the test set
human_true = consult_df["Observed"].to_numpy()      # ground truth labels
human_pred = consult_df["MajorityVote"].to_numpy()  # consultant majority

# quick sanity check
print("First 10 labels, y_test vs human_true:")
print("y_test     :", y_test.values[:10])
print("human_true :", human_true[:10])


# Build a diagnostics DataFrame for the 50 test jurors
diag_df = pd.DataFrame({
    "y_true": y_test.values,
    "Human_pred": human_pred,
    "RF_pred": rf_pred,
    "XGB_OHE_pred": xgb_ohe_pred,
    "Human_correct": (human_pred == y_test.values),
    "RF_correct": (rf_pred == y_test.values),
    "XGB_OHE_correct": (xgb_ohe_pred == y_test.values),
    "RF_proba": rf_proba,
    "XGB_OHE_proba": xgb_ohe_proba,
})

# If Juror # is present in consultant CSV, attach it
for col in ["Juror #", "Juror", "Juror_no", "Juror_ID"]:
    if col in consult_df.columns:
        diag_df[col] = consult_df[col].values
        break  # use the first matching name

diag_df.head(10)


In [ ]:

# HEATMAP: who is correct for each juror?

plot_df = diag_df[["Human_correct", "RF_correct", "XGB_OHE_correct"]].copy()
plot_df = plot_df.replace({True:1, False:0})  # 1 = correct, 0 = wrong
plot_df.index = [f"Juror_{i}" for i in range(len(plot_df))]

plt.figure(figsize=(12, 6))
sns.heatmap(
    plot_df.T,
    cmap="Greens",
    annot=True,
    fmt="d",
    cbar=False
)
plt.title("Correct (1) / Incorrect (0) per Juror Across Models", fontsize=14)
plt.xlabel("Juror")
plt.ylabel("Model")
plt.tight_layout()

plt.savefig("diag_correct_incorrect_heatmap.png", dpi=300, bbox_inches="tight")
plt.savefig("diag_correct_incorrect_heatmap.pdf", bbox_inches="tight")
plt.show()



In [ ]:
# Build pattern_counts from diag_df

# Make sure these columns exist; if not, create them
diag_df["Human_wrong"] = ~diag_df["Human_correct"]
diag_df["RF_wrong"] = ~diag_df["RF_correct"]
diag_df["XGB_OHE_wrong"] = ~diag_df["XGB_OHE_correct"]

# 3-bit pattern string: Human, RF, XGB (1 = wrong, 0 = correct)
diag_df["error_pattern"] = (
    diag_df["Human_wrong"].astype(int).astype(str) +
    diag_df["RF_wrong"].astype(int).astype(str) +
    diag_df["XGB_OHE_wrong"].astype(int).astype(str)
)

pattern_counts = (
    diag_df
    .groupby("error_pattern")
    .size()
    .sort_values(ascending=False)
)

print("Error pattern counts:")
print(pattern_counts)


In [ ]:

# BARPLOT: error pattern frequencies
pattern_order = ["000","100","010","001","110","101","011","111"]
pattern_counts_sorted = pattern_counts.reindex(pattern_order).fillna(0)

plt.figure(figsize=(10,5))
sns.barplot(
    x=pattern_counts_sorted.index,
    y=pattern_counts_sorted.values,
    palette="viridis"
)
plt.title("Error Pattern Frequencies (Human, RF, XGB)", fontsize=14)
plt.xlabel("Error Pattern (H R X)")
plt.ylabel("Juror Count")
plt.tight_layout()

plt.savefig("diag_error_pattern_frequencies.png", dpi=300, bbox_inches="tight")
plt.savefig("diag_error_pattern_frequencies.pdf", bbox_inches="tight")
plt.show()


In [ ]:

# SCATTER: RF vs XGB predicted probabilities (test set)
plt.figure(figsize=(7,7))
sns.scatterplot(
    x=diag_df["RF_proba"],
    y=diag_df["XGB_OHE_proba"],
    hue=diag_df["y_true"],
    palette={0:"red", 1:"blue"},
    s=80
)

plt.axline((0,0),(1,1), color="black", linestyle="--")
plt.title("RF vs XGB Probability Predictions\n(Color = Ground Truth)", fontsize=14)
plt.xlabel("RF Probability (Plaintiff = 1)")
plt.ylabel("XGB Probability (Plaintiff = 1)")
plt.tight_layout()

plt.savefig("diag_rf_vs_xgb_prob_scatter.png", dpi=300, bbox_inches="tight")
plt.savefig("diag_rf_vs_xgb_prob_scatter.pdf", bbox_inches="tight")
plt.show()


In [ ]:

# BARPLOT: how many models got each juror right?

diag_df["num_correct"] = (
      diag_df["Human_correct"].astype(int)
    + diag_df["RF_correct"].astype(int)
    + diag_df["XGB_OHE_correct"].astype(int)
)

plt.figure(figsize=(12,5))
sns.barplot(
    x=list(range(len(diag_df))),
    y=diag_df["num_correct"],
    palette="coolwarm"
)
plt.title("How Many Models Got Each Juror Right?", fontsize=14)
plt.xlabel("Juror")
plt.ylabel("# Correct Predictions (0 to 3)")
plt.ylim(0,3)
plt.tight_layout()

plt.savefig("diag_num_models_correct_per_juror.png", dpi=300, bbox_inches="tight")
plt.savefig("diag_num_models_correct_per_juror.pdf", bbox_inches="tight")
plt.show()


In [ ]:
feature_to_check = "In_a_employment_discrimination_case,_do_you_think_the_plaintiff_i.e.,_the_employee_or_defense_i.e.,_employer_will_have_a_harder_time_convincing_you?_ord"
test_full = X_test.join(
    diag_df[["Human_correct", "RF_correct", "XGB_OHE_correct"]]
)


In [ ]:

# BARPLOT: accuracy by segments for key ordinal feature

feature = feature_to_check

seg = (
    test_full
    .groupby(feature)
    .agg(
        human_acc=("Human_correct","mean"),
        rf_acc=("RF_correct","mean"),
        xgb_acc=("XGB_OHE_correct","mean"),
        count=("Human_correct","size")
    )
    .sort_index()
)

seg_plot = seg[["human_acc","rf_acc","xgb_acc"]]

ax = seg_plot.plot(kind="bar", figsize=(10,5))
plt.title(f"Accuracy by Segment: {feature}", fontsize=14)
plt.ylabel("Accuracy")
plt.xlabel("Ordinal Response Level")
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig("diag_segment_accuracy_by_ordinal_feature.png", dpi=300, bbox_inches="tight")
plt.savefig("diag_segment_accuracy_by_ordinal_feature.pdf", bbox_inches="tight")
plt.show()

print("Segment counts and accuracies:")
print(seg)


In [ ]:
from sklearn.metrics import confusion_matrix

for name, pred, fname in [
    ("Human", human_pred, "cm_human"),
    ("RF", rf_pred, "cm_rf"),
    ("XGB OHE", xgb_ohe_pred, "cm_xgb_ohe"),
]:
    cm = confusion_matrix(y_test, pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, cmap="Blues", fmt="d")
    plt.title(f"Confusion Matrix: {name}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()

    plt.savefig(f"{fname}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{fname}.pdf", bbox_inches="tight")
    plt.show()


In [ ]:

# y_test is the 50 held-out jurors selected via manager_indices from df
y_test_aligned = y_test.reset_index(drop=True)

# Human labels/preds are already in the same order as the 50 test jurors
human_true = consult_df["Observed"].reset_index(drop=True)
human_pred = consult_df["MajorityVote"].reset_index(drop=True)

# Sanity check
print("Lengths -> y_test:", len(y_test_aligned),
      "human_true:", len(human_true),
      "human_pred:", len(human_pred))

assert len(human_true) == len(y_test_aligned), "Human vs test length mismatch!"

# RF predictions
rf_pred = best_rf_base.predict(X_test)
rf_proba = best_rf_base.predict_proba(X_test)[:, 1]

# XGB OHE predictions
xgb_ohe_pred  = best_xgb_ohe.predict(X_test_ohe)
xgb_ohe_proba = best_xgb_ohe.predict_proba(X_test_ohe)[:, 1]

# Build diagnostic dataframe
diag_df = pd.DataFrame({
    "y_true": y_test_aligned,
    "Human_pred": human_pred,
    "RF_pred": rf_pred,
    "RF_proba": rf_proba,
    "XGB_OHE_pred": xgb_ohe_pred,
    "XGB_OHE_proba": xgb_ohe_proba,
})

diag_df["Human_correct"]   = diag_df["Human_pred"]   == diag_df["y_true"]
diag_df["RF_correct"]      = diag_df["RF_pred"]      == diag_df["y_true"]
diag_df["XGB_OHE_correct"] = diag_df["XGB_OHE_pred"] == diag_df["y_true"]

diag_df["pattern"] = (
    diag_df["Human_correct"].astype(int).astype(str) +
    diag_df["RF_correct"].astype(int).astype(str) +
    diag_df["XGB_OHE_correct"].astype(int).astype(str)
)

pattern_counts = diag_df["pattern"].value_counts()
print("Error pattern counts (H,R,X):")
print(pattern_counts)


In [ ]:
# Hard jurors + SHAP explanations (XGB OHE)

import shap

# Hard jurors: all three wrong
hard_mask = (~diag_df["Human_correct"] &
             ~diag_df["RF_correct"] &
             ~diag_df["XGB_OHE_correct"])
hard_indices = diag_df.index[hard_mask].tolist()

print(f"Number of hard jurors (all 3 wrong): {len(hard_indices)}")
print("Hard juror indices in test set:", hard_indices)

# ----- SHAP for XGB OHE -----
# Use TreeExplainer in "new" API mode (returns Explanation)
explainer_xgb = shap.TreeExplainer(best_xgb_ohe)
shap_exp_test = explainer_xgb(X_test_ohe)  # Explanation object

# Global summary plot for context
shap.summary_plot(shap_exp_test, X_test_ohe, show=False)
plt.title("XGB OHE – Global SHAP Summary (Test Set)")
plt.tight_layout()
plt.savefig("shap_xgb_ohe_summary_test.png", dpi=300, bbox_inches="tight")
plt.close()

# Local explanations for each hard juror
for idx in hard_indices:
    print(f"\n=== SHAP for hard juror test-index {idx} ===")
    # Bar plot of top contributing features
    shap.plots.bar(shap_exp_test[idx], max_display=10, show=False)
    plt.title(f"XGB OHE – Top Features for Hard Juror #{idx}")
    plt.tight_layout()
    plt.savefig(f"shap_xgb_ohe_hard_juror_{idx}.png", dpi=300, bbox_inches="tight")
    plt.close()


In [ ]:
# LEARNING CURVE FOR RF & XGB (OHE) ON FIXED TEST SET


import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42

def learning_curve_fixed_test(
    estimator,
    X_train,
    y_train,
    X_test,
    y_test,
    train_sizes,
    n_repeats=10,
    random_state=42,
):
    """
    Learning curve on a fixed test set.

    IMPORTANT:
    - Hyperparameters are FROZEN (taken from `estimator`)
    - A fresh model is trained from scratch for each subset
    - No hyperparameter re-tuning is performed inside the curve

    Returns:
        sizes: np.array of train_sizes
        mean_acc: mean accuracy per size
        std_acc: std dev of accuracy per size
    """
    rng = np.random.RandomState(random_state)
    n_train_total = X_train.shape[0]

    mean_acc = []
    std_acc = []

    for size in train_sizes:
        frac = min(max(size / n_train_total, 0.05), 0.95)

        sss = StratifiedShuffleSplit(
            n_splits=n_repeats,
            train_size=frac,
            random_state=random_state,
        )

        accs = []

        for idx_sub, _ in sss.split(X_train, y_train):

            # Enforce exact train size
            if len(idx_sub) > size:
                idx_sub = rng.choice(idx_sub, size=size, replace=False)

            X_sub = X_train.iloc[idx_sub]
            y_sub = y_train.iloc[idx_sub]

            # Fresh model with FROZEN hyperparameters
            model = estimator.__class__(**estimator.get_params())
            model.fit(X_sub, y_sub)

            y_pred = model.predict(X_test)
            accs.append(accuracy_score(y_test, y_pred))

        accs = np.array(accs)
        mean_acc.append(accs.mean())
        std_acc.append(accs.std())

        print(f"Train size={size:3d} | mean acc={accs.mean():.3f} ± {accs.std():.3f}")

    return np.array(train_sizes), np.array(mean_acc), np.array(std_acc)



# Choose train sizes
n_train = X_train.shape[0]
train_sizes = [50, 80, 110, 150, 200, n_train]

print("Total train size:", n_train)


# RF learning curve (ordinal features)
sizes_rf, mean_rf, std_rf = learning_curve_fixed_test(
    estimator=best_rf_base,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    train_sizes=train_sizes,
    n_repeats=15,
    random_state=RANDOM_STATE,
)


# XGB OHE learning curve (one-hot features)
sizes_xgb, mean_xgb, std_xgb = learning_curve_fixed_test(
    estimator=best_xgb_ohe,
    X_train=X_train_ohe,
    y_train=y_train,
    X_test=X_test_ohe,
    y_test=y_test,
    train_sizes=train_sizes,
    n_repeats=15,
    random_state=RANDOM_STATE,
)


# Plot both learning curves
plt.figure(figsize=(8,6))

plt.plot(sizes_rf, mean_rf, marker="o", label="RF (ordinal)")
plt.fill_between(
    sizes_rf,
    mean_rf - std_rf,
    mean_rf + std_rf,
    alpha=0.2
)

plt.plot(sizes_xgb, mean_xgb, marker="s", label="XGB OHE")
plt.fill_between(
    sizes_xgb,
    mean_xgb - std_xgb,
    mean_xgb + std_xgb,
    alpha=0.2
)

plt.xlabel("Training set size")
plt.ylabel("Accuracy on fixed test set (50 managers)")
plt.title("Learning Curves – RF vs XGB OHE (Mean ± 1 std over repeats)")
plt.legend()
plt.ylim(0.4, 1.0)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("learning_curve_rf_xgb_fixed_test.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved plot as learning_curve_rf_xgb_fixed_test.png")
